### Pinecone Hybrid Search Integration

Pinecone is a vector database with broad functionality.

In [8]:
#importing libraries

import os
from dotenv import load_dotenv

from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder

from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [9]:
index_name = "langchain-pinecone-hybrid-search"

#initializing pinecone client
pc = Pinecone(api_key=api_key)

#creating the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, #dimensionality of dense model
        metric="dotproduct", #sparse values supported only for dotproduct
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

In [10]:
index = pc.Index(index_name)
index

In [11]:
#getting embeddings (used for dense vector) 
embedding = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

In [12]:
#to encode text to sparse vector , we can use SPLADE or BM25, for out of domain tasks we can use BM25.
# it use tf-idf values

bm25_encoder = BM25Encoder().default()

sentences= ["this is good course", "im learning agentic ai", "langchain is very useful to use llms"]

#fitting tf-idf values on corpus

bm25_encoder.fit(sentences)

#store the values to json file
bm25_encoder.dump("bm25_values.json")

#loading bm25_encoder object
bm25_encoder_obj = BM25Encoder().load("bm25_values.json")

100%|██████████| 3/3 [00:00<?, ?it/s]


In [13]:
#constructing retriever

retriever = PineconeHybridSearchRetriever(
    embeddings=embedding,
    sparse_encoder=bm25_encoder,
    index=index
)

retriever


PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000001C4A206EBC0>, index=<pinecone.db_data.index.Index object at 0x000001C4A206D540>)

In [14]:
#adding text in retriever

retriever.add_texts(sentences)


100%|██████████| 1/1 [00:01<00:00,  1.56s/it]


In [18]:
#using hybridsearch retriever

retriever.invoke("langchain")


[Document(metadata={'score': 0.484158099}, page_content='langchain is very useful to use llms'),
 Document(metadata={'score': 0.0986652374}, page_content='this is good course'),
 Document(metadata={'score': 0.057091713}, page_content='im learning agentic ai')]